
# Ejercicios PySpark en Databricks

---

## Ejercicio 1 — Lectura de CSV con esquema definido y formato de fecha

### Contexto

Se te proporciona el siguiente dataset en formato **CSV** con **50 registros** y **6 campos**. El archivo contiene información de ventas con una columna de fecha en formato `dd/MM/yyyy`. Tu tarea es subirlo al volume de Databricks y leerlo correctamente usando PySpark.

---

### Instrucciones

**Paso 1 — Subir el archivo al volume**

Sube el archivo `ventas.csv` al volume de Databricks llamado **`vol_landing`**. Puedes hacerlo desde la interfaz de Databricks:

> `Catalog` → `[Tu catálogo]` → `[Tu schema]` → `vol_landing` → botón **Upload**

La ruta del archivo quedará similar a:

```
/Volumes/<catalogo>/<schema>/vol_landing/ventas.csv
```

---

**Paso 2 — Definir el esquema**

En PySpark, **debes definir explícitamente el esquema** del dataset usando `StructType` y `StructField`. No uses inferencia automática de esquema (`inferSchema=True`).

```python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

# TODO: Completa el esquema con todos los campos del CSV
schema = StructType([
    StructField("___", ___Type(), ___),
    StructField("___", ___Type(), ___),
    StructField("___", ___Type(), ___),
    StructField("___", ___Type(), ___),
    StructField("___", ___Type(), ___),
    StructField("___", ___Type(), ___),
])
```

> 💡 **Pista:** Identifica qué tipo de dato corresponde a cada columna: `id_venta`, `producto`, `categoria`, `cantidad`, `precio_unitario` y `fecha_venta`.

---

**Paso 3 — Leer el CSV con la opción `dateFormat`**

El campo `fecha_venta` está en formato `dd/MM/yyyy`. Para que PySpark lo interprete correctamente como `DateType`, **debes usar la opción `dateFormat`** al momento de la lectura, agregala con el formato correcto.

```python
# TODO: Define la ruta correcta a tu volume
ruta_archivo = "/Volumes/<catalogo>/<schema>/vol_landing/ventas.csv"

# TODO: Completa las opciones de lectura
df_ventas = spark.read \
    .format("csv") \
    .option("header", ___) \
    .schema(schema) \
    .load(ruta_archivo)

df_ventas.display()

```

---

**Paso 4 — Validaciones**

Una vez leído el archivo, responde las siguientes preguntas usando transformaciones PySpark:

1. ¿Cuántos registros tiene el DataFrame?
2. ¿Cuál es el total de ingresos por categoría? *(Pista: `cantidad * precio_unitario`)*
3. ¿Cuál es la fecha de venta más reciente y la más antigua del dataset?
4. Filtra únicamente las ventas del mes de **febrero de 2024**.
5. Escriba los datos en una tabla administrada en el catalogo y esquema de su preferencia.
---


## Resolución Ejercicio n1

In [0]:
VOLUME_PATH    = "/Volumes/dbassociate/default/vol_landing"
CSV_VENTAS = f"{VOLUME_PATH}/ventas.csv"

for ruta in [CSV_VENTAS]:
    try:
        dbutils.fs.ls(ruta)
        print(f"el archivo con ruta -> : {ruta} si existe")
    except Exception:
        print(f"NO ENCONTRADO : {ruta}  <-- subir el archivo al Volume")

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

# Opcion 1: inferencia de esquema
# Antipatron en produccion: Spark lee el archivo DOS veces (1x para inferir, 1x para cargar)
df_ventas_inferred = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(CSV_VENTAS)
)

print("=== Esquema INFERIDO ===")
df_ventas_inferred.printSchema()

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType
)

# Opcion 2: esquema explicito (practica recomendada para produccion)
# Una sola pasada de lectura, comportamiento predecible
schema_ventas = StructType([
    StructField("id_venta",  StringType(),  nullable=False),
    StructField("producto",      StringType(),  nullable=True),
    StructField("categoria",  StringType(),  nullable=True),
    StructField("cantidad",        IntegerType(), nullable=True),
    StructField("precio_unitario", DoubleType(),  nullable=True),
    StructField("fecha_venta",           DateType(),  nullable=True)
    
])

df_ventas = (
    spark.read
    .option("header", "true")
    .option("dateFormat", "dd/MM/yyyy")
    .schema(schema_ventas)
    .csv(CSV_VENTAS)
)

print("=== Esquema EXPLICITO ===")
df_ventas.printSchema()
display(df_ventas.limit(5))

In [0]:
##EJERCICIO 5 ->
##/Volumes/dbassociate/default/vol_landing/empleados.json
# Guardamos tu DataFrame en la ruta de 3 niveles que preparaste
(df_ventas.write
 .mode("overwrite") # Reemplaza la tabla si vuelves a correr la celda
 .saveAsTable("bda_certificacion.sales.ventas_procesadas"))

print("¡Listo! Tu DataFrame ahora es una Tabla Administrada en Unity Catalog.")

In [0]:
df_ventas.count()

In [0]:
from pyspark.sql.functions import col, sum

df_ingresos = df_ventas.withColumn("ingreso_total", col("cantidad") * col("precio_unitario")) \
                       .groupBy("categoria") \
                       .agg(sum("ingreso_total").alias("total_ingresos"))

display(df_ingresos)

In [0]:
from pyspark.sql.functions import min, max

df_fechas = df_ventas.agg(
    min("fecha_venta").alias("fecha_mas_antigua"),
    max("fecha_venta").alias("fecha_mas_reciente")
)

display(df_fechas)

In [0]:
from pyspark.sql.functions import month, year

# Recuerda usar paréntesis obligatorios para cada condición cuando uses el &
df_febrero = df_ventas.filter((month(col("fecha_venta")) == 2) & (year(col("fecha_venta")) == 2024))

display(df_febrero)


## Ejercicio 2 — Lectura de JSON Multilínea con esquema definido

### Contexto

Se te proporciona un archivo **JSON de 50 registros** con estructura **multilinea**. Este tipo de archivo no puede leerse con la configuración por defecto de PySpark. Tu tarea es subirlo al volume y leerlo correctamente usando la opción `multiLine`.

---

### Instrucciones

**Paso 1 — Subir el archivo al volume**

Sube el archivo `empleados.json` al volume de Databricks llamado **`vol_landing`**. Puedes hacerlo desde la interfaz de Databricks:

> `Catalog` → `[Tu catálogo]` → `[Tu schema]` → `vol_landing` → botón **Upload**

La ruta del archivo quedará similar a:

```
/Volumes/<catalogo>/<schema>/vol_landing/empleados.json
```

---

**Paso 2 — Definir el esquema**

En PySpark, **debes definir explícitamente el esquema** del dataset usando `StructType` y `StructField`. No uses inferencia automática de esquema.

```python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType

# TODO: Completa el esquema con todos los campos del JSON
schema = StructType([
    StructField("___", ___Type(), ___),
    StructField("___", ___Type(), ___),
    StructField("___", ___Type(), ___),
    StructField("___", ___Type(), ___),
    StructField("___", ___Type(), ___),
    StructField("___", ___Type(), ___),
])
```

> 💡 **Pista:** Revisa los campos del JSON: `id_empleado`, `nombre`, `departamento`, `cargo`, `salario` y `activo`. Presta atención al tipo booleano.

---

**Paso 3 — Leer el JSON con la opción `multiLine`**

El archivo JSON es de tipo **array multilinea** (comienza con `[` y cierra con `]`). PySpark, por defecto, espera un JSON por línea (formato JSONL). Si intentas leerlo sin la opción correcta, obtendrás un error o un DataFrame vacío.

**Debes usar la opción `multiLine` para leerlo correctamente.**

```python
# TODO: Define la ruta correcta a tu volume
ruta_archivo = "/Volumes/<catalogo>/<schema>/vol_landing/empleados.json"

# TODO: Completa las opciones de lectura
df_empleados = spark.read \
    .format("json") \
    .schema(schema) \
    .load(ruta_archivo)

df_empleados.display()
```

---

**Paso 4 — Validaciones**

Una vez leído el archivo, responde las siguientes preguntas usando transformaciones PySpark:

1. ¿Cuántos empleados están activos y cuántos inactivos?
2. ¿Cuál es el salario promedio por departamento?
3. ¿Cuál es el empleado con el salario más alto de todo el dataset?
4. Lista todos los empleados del departamento de **Tecnología** ordenados por salario de mayor a menor.
5. Escriba los datos en una tabla administrada en el catalogo y esquema de su preferencia.


## Resolución Ejercicio n1

In [0]:
vol_landing = '/Volumes/dbassociate/default/vol_landing'
empleados = f"{vol_landing}/empleados.json"
print(empleados)

In [0]:
df_empleados_v2 = spark.read.json(empleados)
df_empleados_v2.printSchema()
display(df_empleados_v2.limit(3))

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType
schema_empleados = StructType([
    StructField("id_empleado", StringType(), True),
    StructField("nombre", StringType(), True),
    StructField("departamento", StringType(), True),
    StructField("cargo", StringType(), True),
    StructField("salario", DoubleType(), True),
    StructField("activo", StringType())
                
])

df_empleados_v2 = spark.read.schema(schema_empleados).json(empleados)
df_empleados_v2.printSchema()
display(df_empleados_v2.limit(3))

In [0]:
##EJERCICIO 1 ->
from pyspark.sql import functions as F

df_conteo = df_empleados_v2.groupBy("activo").agg(F.count("*").alias("conteo"))
display(df_conteo)

In [0]:
##EJERCICIO 2 ->
from pyspark.sql import functions as F
df_avg = df_empleados_v2.groupBy("departamento").agg(F.avg("salario").alias("salario_promedio"))
display(df_avg)

In [0]:
##EJERCICIO 3 ->
from pyspark.sql import functions as F

df_max = df_empleados_v2.agg(F.max("salario")).alias("salario_maximo")
display(df_max)

In [0]:
##EJERCICIO 4 ->
from pyspark.sql import functions as F
df_tecno = (df_empleados_v2
            .filter(F.col("departamento") == "Tecnología")
            .orderBy(F.col("salario").desc())
            )
display(df_tecno)

In [0]:
##EJERCICIO 5 ->
##/Volumes/dbassociate/default/vol_landing/empleados.json
# Guardamos tu DataFrame en la ruta de 3 niveles que preparaste
(df_empleados_v2.write
 .mode("overwrite") # Reemplaza la tabla si vuelves a correr la celda
 .saveAsTable("bda_certificacion.rrhh.empleados_procesados"))

print("¡Listo! Tu DataFrame ahora es una Tabla Administrada en Unity Catalog.")